<style>
.jp-RenderedHTMLCommon,
.text_cell_render {
  font-family: medium-content-serif-font, Georgia, Cambria, "Times New Roman", Times, serif;
  font-size: 17px;
  line-height: 1.65;
  color: #1f2328;
}
.jp-RenderedHTMLCommon h1,
.jp-RenderedHTMLCommon h2,
.jp-RenderedHTMLCommon h3,
.text_cell_render h1,
.text_cell_render h2,
.text_cell_render h3 {
  font-family: sohne, "Helvetica Neue", Helvetica, Arial, sans-serif;
  color: #111;
}
.jp-RenderedHTMLCommon h1,
.text_cell_render h1 {
  font-size: 2.6rem;
  letter-spacing: -0.03em;
}
.jp-RenderedHTMLCommon h2,
.text_cell_render h2 {
  margin-top: 1.8em;
}
</style>

# ¿Qué hace que una canción sea un éxito?

Data Analytics, Semana 9: Regresión e IA para analítica, 2026-2

Este notebook acompaña la lectura `Blog_DA_S9_Regresion_Canciones.html`. La lectura desarrolla los conceptos; aquí se ejecuta el análisis paso a paso sobre el mismo caso de estudio.

Imagine que una disquera debe decidir, antes de lanzar una canción, cuáles títulos merecen una mayor inversión promocional. Para apoyar esa decisión se quiere estimar la `popularidad` que alcanzará cada canción: un valor continuo, es decir, una tarea de regresión.

El trabajo sigue cinco fases: exploración y calidad de los datos, preparación de variables y control de fuga, modelado con tres algoritmos de complejidad creciente, evaluación con métricas de regresión, e importancia de características con una conclusión para el negocio. Con esa misma vara se revisa, al final, un resultado generado por un asistente de IA.

El objetivo no es obtener la métrica más alta posible. Es construir un modelo que resista una auditoría: que solo use información disponible en el momento de decidir, que se compare con una referencia simple, que se evalúe sobre canciones posteriores a las que vio y que deje registro de cada intento. Con esa misma vara se revisará, al final, un resultado generado por un asistente de IA.

## Preparar los datos

El notebook lee `data/canciones.csv` del repositorio y puede ejecutarse desde la carpeta `notebooks/` o desde la raíz. Cada fila representa una canción sintética.

Las filas se ordenan por año de lanzamiento. Ese orden se usará más adelante para separar el pasado del futuro.

In [ ]:
from pathlib import Path
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = RAIZ / 'data' / 'canciones.csv'
REPORTES = RAIZ / 'reportes'
REPORTES.mkdir(exist_ok=True)

canciones = pd.read_csv(DATA_PATH)
canciones = canciones.sort_values('anio', kind='stable')
canciones = canciones.reset_index(drop=True)
print(canciones.shape)
canciones.head()

### El momento de la decisión

La disquera decide **antes del lanzamiento**. Esa frase define qué información es legítima: una columna puede estar en el archivo y, aun así, no existir todavía en el momento en que se necesita la predicción.

Antes de modelar, complete esta tabla de control en su cuaderno o en una celda de texto. Es el primer entregable de cualquier ejercicio de *feature engineering*.

| Columna | Qué representa | ¿Se conoce antes del lanzamiento? | Decisión |
|---|---|---|---|
| `bailabilidad`, `energia`, `valencia`, `acustica` | | | |
| `tempo`, `duracion_min`, `volumen_db` | | | |
| `anio`, `colaboracion`, `genero` | | | |
| `reproducciones_sem1` | | | |
| `es_hit` | | | |
| `popularidad` | Variable objetivo | | Objetivo |

In [ ]:
objetivo = 'popularidad'
cat = ['genero']
todas = [c for c in canciones.columns if c != objetivo]
y = canciones[objetivo]
print(len(todas), 'columnas candidatas:')
print(textwrap.fill(', '.join(todas), 58))

## Antes de entrenar, separar el pasado del futuro

El modelo se usará siempre sobre canciones que todavía no existen. La evaluación debe imitar ese uso: se entrena con las canciones lanzadas hasta 2019 y se prueba con las lanzadas entre 2020 y 2025.

Esta partición se hace antes de explorar. El conjunto de prueba se conserva para observar qué ocurre con canciones que no participaron en ninguna decisión del análisis.

In [ ]:
train = canciones['anio'] < 2020
test = ~train
print('train', train.sum(), '| test', test.sum())
print(canciones.loc[train, 'anio'].max(), '<',
      canciones.loc[test, 'anio'].min())

## Explorar sin mirar la prueba

La primera revisión de distribuciones y relaciones entre variables se realiza únicamente sobre entrenamiento. Así se evita que la información del conjunto de prueba influya en decisiones tempranas.

In [ ]:
canciones[train].describe().round(2)

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(y[train], bins=30)
plt.xlabel('popularidad')
plt.ylabel('frecuencia')
plt.title('Distribución de popularidad en entrenamiento')
plt.show()

### Calidad de los datos: nulos, duplicados y valores extremos

Antes de modelar se revisa qué tan limpio está el conjunto. Los valores extremos se marcan con la regla de 1,5 × IQR sobre entrenamiento; marcarlos no significa eliminarlos. En este curso un valor extremo se investiga y se decide con un criterio explícito, como se hizo en el EDA univariado.

Observe qué columnas aparecen en la lista y pregúntese si en todas tiene sentido hablar de «valor extremo».

In [ ]:
numericas = canciones[train].select_dtypes('number')
q1, q3 = numericas.quantile(.25), numericas.quantile(.75)
iqr = q3 - q1
fuera = (numericas < q1 - 1.5 * iqr) | (numericas > q3 + 1.5 * iqr)
print('nulos:', int(canciones.isna().sum().sum()),
      '| duplicados:', int(canciones.duplicated().sum()))
extremos = fuera.sum()
print('extremos (1,5 × IQR, entrenamiento):')
for col, n in extremos[extremos > 0].items():
    print(f'{col:<20} {n:>4}')

In [ ]:
correlaciones = (
    canciones[train].select_dtypes('number')
    .corr()[objetivo].drop(objetivo).sort_values()
)

plt.figure(figsize=(7, 5))
correlaciones.plot(kind='barh')
plt.xlabel('correlación de Pearson')
plt.title('Correlación con popularidad (entrenamiento)')
plt.show()

print(correlaciones.round(3))
print()
print(canciones[train].groupby('genero')[objetivo]
      .agg(['size', 'mean']).round(1))

### Correlación entre variables explicativas

La matriz anterior mira cada variable frente al objetivo. Esta mira las variables entre sí: dos variables muy correlacionadas aportan casi la misma información (multicolinealidad), hacen inestables los coeficientes de un modelo lineal y dificultan la lectura de la importancia de cada una.

In [ ]:
acusticas = ['bailabilidad', 'energia', 'valencia', 'acustica',
             'tempo', 'duracion_min', 'volumen_db']
entre = canciones.loc[train, acusticas].corr()
mascara = ~np.eye(len(acusticas), dtype=bool)
print('|r| máximo entre variables explicativas:',
      round(entre.where(mascara).abs().max().max(), 2))

plt.figure(figsize=(6.5, 5.2))
plt.imshow(entre, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(label='correlación de Pearson')
plt.xticks(range(len(acusticas)), acusticas, rotation=45, ha='right')
plt.yticks(range(len(acusticas)), acusticas)
plt.title('Correlación entre variables explicativas (entrenamiento)')
plt.tight_layout()
plt.show()

### Antes de seguir

Responda a partir de las salidas anteriores:

1. ¿Qué variables parecen tener una relación más fuerte con `popularidad`?
2. ¿Alguna relación es tan fuerte que resulta sospechosa?
3. ¿Hay alguna variable cuya disponibilidad temporal deba revisarse antes de utilizarla para predecir?
4. ¿Hay alguna columna que parezca calculada a partir de la propia variable objetivo?
5. ¿Hay pares de variables explicativas redundantes? ¿Qué haría si los hubiera?
6. ¿Por qué `es_hit` aparece con 305 «valores extremos»? ¿Qué dice eso sobre aplicar la regla IQR a una variable binaria?

Una correlación alta puede ser informativa, pero no basta para decidir que una variable debe entrar al modelo. Antes de utilizar cualquier variable, revise dos aspectos: qué relación muestra con el objetivo y en qué momento estaría disponible.

## Una línea base y un registro antes de hablar de un buen modelo

La línea base funciona como una referencia mínima: predecir siempre la media observada en entrenamiento. Una segunda referencia, un poco más exigente, predice la media de cada género.

La función `evaluar` calcula las métricas sobre el conjunto de prueba y guarda cada intento en `registro`. Ese registro comparativo es uno de los entregables de la semana: todo lo que se pruebe queda anotado, también lo que después se descarte.

In [ ]:
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import r2_score

registro = []
def evaluar(corrida, pred, real=None, validacion='temporal'):
    real = y[test] if real is None else real
    fila = dict(corrida=corrida, validacion=validacion,
                MAE=round(mae(real, pred), 2),
                RMSE=round(mse(real, pred) ** 0.5, 2),
                R2=round(r2_score(real, pred), 3))
    registro.append(fila)
    print(f'{corrida:<22}',
          'MAE %5.2f  R2 %6.3f' % (fila['MAE'], fila['R2']))

In [ ]:
genero_train = canciones.loc[train, 'genero']
media_genero = y[train].groupby(genero_train).mean()
evaluar('media global', np.full(test.sum(), y[train].mean()))
evaluar('media por género',
        canciones.loc[test, 'genero'].map(media_genero))

## Cuando un modelo parece demasiado bueno

A continuación se ajusta una regresión lineal con todas las columnas candidatas. El preprocesamiento vive dentro de un `Pipeline`: la codificación de `genero` y el escalado de las variables numéricas se aprenden únicamente con entrenamiento.

`armar` recibe cualquier estimador de scikit-learn y devuelve el flujo completo. `correr` lo entrena con el pasado, lo prueba en el futuro y deja el resultado en el registro.

Observe el resultado antes de continuar. Un desempeño excepcional en un primer intento no debe interpretarse automáticamente como evidencia de un buen modelo.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

def armar(estimador, columnas):
    categoricas = [c for c in columnas if c in cat]
    numericas = [c for c in columnas if c not in cat]
    codificar = OneHotEncoder(handle_unknown='ignore',
                              sparse_output=False)
    pre = ColumnTransformer([('cat', codificar, categoricas),
                             ('num', StandardScaler(), numericas)])
    return make_pipeline(pre, estimador)

In [ ]:
def correr(corrida, columnas, estimador=None):
    if estimador is None:
        estimador = LinearRegression()
    modelo = armar(estimador, columnas)
    modelo.fit(canciones.loc[train, columnas], y[train])
    pred = modelo.predict(canciones.loc[test, columnas])
    evaluar(corrida, pred)
    return modelo, pred

modelo_todas, pred_todas = correr('lineal, todas', todas)

### Investigar las variables sospechosas

El resultado anterior debe compararse con la evidencia obtenida durante la exploración y con la tabla de control que usted completó.

Identifique las columnas cuya inclusión puede estar generando una evaluación sospechosamente optimista. Para sustentar la elección, considere:

- su relación con `popularidad`;
- qué representa cada columna;
- en qué momento se conocería su valor;
- si esa información existiría antes de decidir cuánto invertir en la promoción de una canción;
- si la columna se calcula a partir del propio objetivo.

Escriba los nombres exactos en la lista `fuera_del_modelo`. A diferencia de una respuesta tecleada al ejecutar, la lista queda escrita en el código: se versiona, se revisa y hace reproducible el notebook completo.

In [ ]:
# Complete la lista con los nombres exactos de las columnas que deben quedar fuera.
fuera_del_modelo = []

if not fuera_del_modelo:
    raise ValueError(
        "La lista 'fuera_del_modelo' está vacía. Revise la exploración, complete "
        "la lista y vuelva a ejecutar esta celda.")
desconocidas = [c for c in fuera_del_modelo if c not in todas]
if desconocidas:
    raise ValueError(f'No son columnas candidatas: {desconocidas}')

validas = [c for c in todas if c not in fuera_del_modelo]
print('Fuera del modelo:', fuera_del_modelo)
print('Variables válidas:', len(validas))

## Regresión: medir cuánto se equivoca el modelo

Se vuelve a ajustar exactamente el mismo modelo después de retirar únicamente las columnas seleccionadas en el diagnóstico.

La comparación relevante es triple: frente a la línea base, frente al modelo con todas las columnas y frente al nuevo modelo. Si el cambio en las métricas es grande, deberá explicarse por qué el modelo inicial parecía tan bueno y si esa ventaja era válida para el escenario planteado.

In [ ]:
modelo, pred = correr('lineal, válidas', validas)

### Leer las métricas de regresión

Para una observación, el error puede escribirse como `e = y - ŷ`.

- MAE es el promedio de `|y - ŷ|`. Queda en la misma escala de `popularidad`.
- MSE es el promedio de `(y - ŷ)²`. Penaliza con más fuerza los errores grandes.
- RMSE es `√MSE`. Mantiene esa penalización y vuelve a la escala original.
- R² compara el modelo con la referencia de predecir siempre la media.

MAE y RMSE describen el tamaño del error; R² aporta una lectura relativa frente a una referencia. Ninguna de estas medidas debe interpretarse de forma aislada, y ninguna significa «porcentaje de aciertos».

In [ ]:
errores = np.array([1, 2, 1, 2, 15])

print(pd.DataFrame({'error': errores,
                    'error_absoluto': np.abs(errores),
                    'error_cuadrado': errores ** 2}))
print('MAE: ', round(np.mean(np.abs(errores)), 2))
print('MSE: ', round(np.mean(errores ** 2), 2))
print('RMSE:', round(np.sqrt(np.mean(errores ** 2)), 2))

In [ ]:
residuos = y[test] - pred

plt.figure(figsize=(6, 5))
plt.scatter(y[test], pred, alpha=0.45, s=16)
limites = [min(y[test].min(), pred.min()), max(y[test].max(), pred.max())]
plt.plot(limites, limites, linestyle='--')
plt.xlabel('popularidad real')
plt.ylabel('popularidad esperada')
plt.title('Valores reales y esperados (2020-2025)')
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(residuos, bins=30)
plt.axvline(0, linestyle='--')
plt.xlabel('residuo')
plt.ylabel('frecuencia')
plt.title('Distribución de residuos')
plt.show()

print('dentro de ±10 puntos:', round((residuos.abs() <= 10).mean(), 3))

### Deténgase y revise

- ¿Cuánto cambian MAE, RMSE y R² al retirar las columnas seleccionadas?
- ¿Ese cambio respalda la hipótesis de una evaluación artificialmente optimista?
- ¿El modelo corregido mejora frente a las dos líneas base? ¿En cuántos puntos de popularidad?
- ¿Qué diferencia observa entre MAE y RMSE?
- ¿Qué muestran los residuos sobre los casos en los que el modelo se equivoca más?

## Feature engineering: una variable nueva se mide, no se supone

Preparar variables no es solo transformarlas: es decidir cuáles entran y comprobar si aportan. Aquí se prueban tres ideas sobre el mismo modelo y con la misma validación:

- retirar `genero`, para medir cuánto aporta la codificación de la variable categórica;
- agregar un término cuadrático para la duración, porque las canciones muy cortas o muy largas podrían comportarse distinto;
- agregar una interacción entre `bailabilidad` y `energia`.

Las columnas nuevas son transformaciones fila a fila con constantes fijadas de antemano: no aprenden nada de las demás canciones, por eso pueden calcularse sobre toda la tabla. Todo lo que sí se aprende de los datos (medias, escalas, categorías) queda dentro del `Pipeline`.

In [ ]:
dur = canciones['duracion_min']
canciones['duracion_c2'] = (dur - 3.5) ** 2
canciones['baile_x_energia'] = (canciones['bailabilidad']
                                * canciones['energia'])

sin_genero = [c for c in validas if c != 'genero']
con_inter = validas + ['baile_x_energia']
finales = validas + ['duracion_c2']
_ = correr('válidas sin genero', sin_genero)
_ = correr('válidas + interacción', con_inter)
modelo, pred = correr('válidas + duración²', finales)

## Validar como se va a usar

Dentro del entrenamiento se usa una validación interna de ventana creciente: cada pliegue ajusta el modelo con las canciones más antiguas y lo valida en el bloque siguiente. Con esta cifra se eligen variables y modelos; el conjunto 2020-2025 se reserva para el final.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import cross_val_score

# las filas ya están ordenadas por anio
cv = TimeSeriesSplit(n_splits=4)
puntajes = cross_val_score(armar(LinearRegression(), finales),
                           canciones.loc[train, finales],
                           y[train], cv=cv,
                           scoring='neg_mean_absolute_error')
mae_cv = -puntajes
print(mae_cv.round(2), '| media =', round(mae_cv.mean(), 2))

## Modelado: tres algoritmos de complejidad creciente

Con las mismas variables y la misma validación se comparan tres algoritmos: la regresión lineal ya ajustada, un bosque aleatorio (`RandomForestRegressor`) y un *gradient boosting* de histogramas (`HistGradientBoostingRegressor`). Cambiar de algoritmo cuesta una línea, porque `armar` recibe cualquier estimador de scikit-learn.

Un modelo más complejo no es automáticamente mejor. Aquí se comparan con dos preguntas: cuánto se equivocan en 2020-2025 y hacia dónde.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import (
    HistGradientBoostingRegressor as HGB)

bosque = RandomForestRegressor(n_estimators=300,
                               random_state=SEED, n_jobs=-1)
_, pred_rf = correr('Random Forest', finales, bosque)
_, pred_hgb = correr('HGB', finales, HGB(random_state=SEED))
print('sesgo lineal %.2f | RF %.2f | HGB %.2f' % (
    (y[test] - pred).mean(), (y[test] - pred_rf).mean(),
    (y[test] - pred_hgb).mean()))

### Deténgase y revise

- ¿Cuál de los tres algoritmos elegiría y por qué? Responda con el registro, no con la fama del algoritmo.
- Los dos modelos de árboles tienen un sesgo positivo cercano a 2 puntos en 2020-2025: subestiman. La regresión lineal, 0,3. ¿Qué tiene el año de lanzamiento que un árbol no puede extrapolar?

### La misma pregunta con partición aleatoria

¿Qué habría pasado con una partición al azar, como la de casi todos los tutoriales? Se mide con dos de los modelos.

In [ ]:
from sklearn.model_selection import train_test_split

Xa, Xb, ya, yb = train_test_split(canciones[finales], y,
                                  test_size=test.mean(),
                                  random_state=SEED)
for nombre, est in [('lineal (al azar)', LinearRegression()),
                    ('HGB (al azar)', HGB(random_state=SEED))]:
    m = armar(est, finales).fit(Xa, ya)
    evaluar(nombre, m.predict(Xb), real=yb,
            validacion='aleatoria')

### Deténgase y revise

- ¿La partición aleatoria resulta más optimista que la temporal en este conjunto de datos? Responda con las cifras del registro, no con lo que esperaba encontrar.
- ¿Por qué, aun si las cifras se parecen, la partición temporal sigue siendo la que corresponde a este caso?

## El error no se reparte igual

Las métricas globales resumen el comportamiento del modelo, pero no dicen dónde falla. Revise el error por género y los casos con mayor error. Para cada caso, pregúntese qué información podría faltar y qué variable adicional sería posible conocer antes del lanzamiento.

In [ ]:
residuos = y[test] - pred
res = pd.DataFrame({'genero': canciones.loc[test, 'genero'],
                    'residuo': residuos,
                    'abs': residuos.abs()})
tabla = res.groupby('genero').agg(
    n=('abs', 'size'), MAE=('abs', 'mean'),
    sesgo=('residuo', 'mean'))
print(tabla.round(2))
print('sesgo lineal =', round(residuos.mean(), 2),
      '| sesgo HGB =', round((y[test] - pred_hgb).mean(), 2))

In [ ]:
peores = canciones.loc[test, validas].copy()
peores['popularidad_real'] = y[test]
peores['popularidad_esperada'] = pred.round(1)
peores['error_absoluto'] = residuos.abs().round(1)
peores.sort_values('error_absoluto', ascending=False).head(5)

## Importancia de características y conclusión de negocio

La disquera no quiere solo un número: quiere saber qué atributos de una canción pesan más en su popularidad. Dos lecturas complementarias sobre el modelo seleccionado:

- **Importancia por permutación.** Se baraja una variable a la vez en el conjunto de prueba y se mide cuánto empeora el MAE. Funciona con cualquier modelo y se lee en puntos de popularidad.
- **Coeficientes del modelo lineal.** Dicen además la dirección del efecto. Como las variables numéricas están escaladas, cada coeficiente es el efecto de una desviación estándar; los de `genero`, el efecto de pertenecer a ese género.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    modelo, canciones.loc[test, finales], y[test],
    scoring='neg_mean_absolute_error',
    n_repeats=10, random_state=SEED)
importancia = pd.Series(perm.importances_mean, index=finales)
importancia = importancia.sort_values(ascending=False)
print('aumento del MAE al barajar cada variable:')
print(importancia.round(2).to_string())

In [ ]:
nombres = modelo[0].get_feature_names_out()
coefs = pd.Series(modelo[-1].coef_, index=nombres).sort_values()
print('efecto en puntos de popularidad por +1 desviación')
print('estándar (numéricas) o por pertenecer al género:')
print(coefs.round(2))

### Deténgase y revise

- ¿Qué atributos acústicos pesan más? ¿La `energia` y la `bailabilidad` predicen mejor que la duración o el tempo?
- ¿Coinciden las dos lecturas? ¿En qué se diferencian?
- Un coeficiente de −5,3 para `indie` no significa que el indie «cause» menos popularidad. ¿Qué otras explicaciones hay? Recuerde la Semana 8.
- Escriba en dos frases lo que le diría a la disquera sobre qué hace popular a una canción, con sus límites.

## Del error a la decisión

El modelo entrega un número; la disquera necesita una lista de canciones para promocionar. Convertir lo primero en lo segundo exige fijar un corte: se promocionan las canciones cuya popularidad esperada supera cierto valor.

Se toma como referencia una popularidad real de 70 puntos. Observe cómo cambia, según el corte, cuántas canciones se promocionan, cuántas de ellas alcanzaron los 70 y cuántas canciones que sí los alcanzaron se quedaron sin promoción.

In [ ]:
meta = y[test] >= 70
print('alcanzaron 70 puntos:', meta.sum(), 'de', test.sum())
for corte in (60, 65, 70):
    promo = pred >= corte
    print(f'corte {corte}: {promo.sum():>3} promocionadas,',
          f'{(promo & meta).sum():>3} llegaron a 70,',
          f'{(~promo & meta).sum():>2} sin promoción')

### Deténgase y revise

- ¿Cuál corte elegiría si el presupuesto alcanza para promocionar 100 canciones?
- ¿Qué cuesta más a la disquera: promocionar una canción que no despega o dejar sin promoción una que habría despegado?
- ¿Quién debería fijar el corte: quien construye el modelo o quien responde por el presupuesto? ¿Con qué información?

## El registro comparativo

Todas las corridas de la sesión, con su forma de validación. El archivo se guarda junto al notebook para que quede en el repositorio.

In [ ]:
tabla_registro = pd.DataFrame(registro)
tabla_registro.to_csv(REPORTES / 'registro_semana9.csv', index=False)
print(tabla_registro.to_string(index=False))

## Lectura crítica de un resultado generado por IA

A un asistente de IA se le entregó el archivo `canciones.csv` con este encargo: *«Entrena un modelo que prediga la popularidad y dime qué tan bueno es.»* Devolvió el código de la celda siguiente y esta respuesta:

> «El modelo de regresión lineal alcanza un R² de 0,98, es decir, una precisión del 98 %. Las variables más importantes son las reproducciones de la primera semana y el indicador de hit. El modelo generaliza bien y está listo para estimar la popularidad de canciones nuevas.»

El código corre, usa librerías estándar y el número es reproducible. Justamente por eso conviene auditarlo con la misma vara que se aplicó al modelo propio.

Este es un caso didáctico: la respuesta fue redactada para el curso con errores frecuentes en el código que producen los asistentes. En la actividad entregable usted auditará la salida de su propio asistente.

In [ ]:
# --- código entregado por el asistente de IA ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

datos = pd.read_csv(DATA_PATH)
X_ia = pd.get_dummies(datos.drop(columns='popularidad'),
                      dtype=float)
X_ia = StandardScaler().fit_transform(X_ia)
y_ia = datos['popularidad']
Xtr, Xte, ytr, yte = train_test_split(
    X_ia, y_ia, test_size=0.2, random_state=42)
modelo_ia = LinearRegression().fit(Xtr, ytr)
print('R2 =', round(modelo_ia.score(Xte, yte), 3))

### Cinco preguntas, en este orden

| Pregunta | Qué se revisa | Cómo se comprueba |
|---|---|---|
| 1. ¿Corre y da lo mismo? | Reproducibilidad | Ejecutar el código tal cual, con semilla fija |
| 2. ¿Qué sabía el modelo? | Fuga de información | Cada columna contra `fuera_del_modelo`; quitar las sospechosas y volver a medir |
| 3. ¿Cómo se validó? | Partición | Repetir la evaluación separando el pasado del futuro |
| 4. ¿Contra qué se compara? | Línea base y métrica | Calcular la línea base con la misma métrica; pasar el error a puntos de popularidad |
| 5. ¿Qué afirma de más? | Interpretación | Separar lo medido de lo concluido: «precisión», importancia, generalización |

La función `auditar` conserva el modelo de la IA y cambia una sola cosa a la vez, para aislar el efecto de cada corrección.

In [ ]:
def auditar(quitar, temporal):
    d = datos.drop(columns=['popularidad'] + quitar)
    Xi = pd.get_dummies(d, dtype=float).to_numpy()
    yi = datos['popularidad'].to_numpy()
    a, b = train_test_split(np.arange(len(datos)),
                            test_size=0.2, random_state=42)
    if temporal:
        a = np.where(datos['anio'] < 2020)[0]
        b = np.where(datos['anio'] >= 2020)[0]
    p = LinearRegression().fit(Xi[a], yi[a]).predict(Xi[b])
    base = np.full(len(b), yi[a].mean())
    return (r2_score(yi[b], p), mae(yi[b], p), mae(yi[b], base))

In [ ]:
pasos = [('1. tal como se entregó', [], False),
         ('2. sin las columnas con fuga', fuera_del_modelo, False),
         ('3. y separando pasado y futuro', fuera_del_modelo, True)]
for paso, quitar, temporal in pasos:
    print(f'{paso:<31}', 'R2 %.2f  MAE %5.2f  (línea base %5.2f)'
          % auditar(quitar, temporal))

### Ficha de auditoría

Complete la evidencia y el veredicto de cada afirmación. Cada veredicto debe apoyarse en una salida ejecutable de este notebook.

| Afirmación de la IA | Evidencia de la verificación | Veredicto |
|---|---|---|
| «Alcanza un R² de 0,98» | | |
| «Una precisión del 98 %» | | |
| «Las variables más importantes son las reproducciones de la primera semana y el indicador de hit» | | |
| «Generaliza bien» | | |
| «Está listo para estimar la popularidad de canciones nuevas» | | |

## Del notebook al servicio

El modelo seleccionado se empaqueta con su ficha para que el servicio lo use. `src/entrenar_regresion.py` reproduce el pipeline de este notebook a partir de las columnas crudas de entrada; la variable `duracion_c2` se calcula dentro del pipeline con `src.variables.derivadas`, de modo que el servicio no necesita recibirla.

La celda siguiente escribe `models/modelo.joblib` y compara el MAE de la ficha con el del registro comparativo. Si las dos cifras no coinciden, el artefacto no corresponde al modelo que se analizó.

In [ ]:
import sys

sys.path.insert(0, str(RAIZ))
from src.entrenar_regresion import entrenar

ficha = entrenar()['ficha']
mae_registro = tabla_registro.loc[tabla_registro['corrida'] == 'válidas + duración²', 'MAE'].iloc[0]
print('MAE en la ficha:', ficha['desempeno']['mae'], '| MAE en el registro:', mae_registro)
print('línea base:', ficha['linea_base']['mae'], '| corte de promoción:', ficha['decision']['corte_promocion'])

## Actividad entregable

Cada estudiante actúa como analista de datos de la disquera y entrega, en su fork de este repositorio, un notebook ejecutable en `notebooks/` y un informe breve organizados en las cinco fases del trabajo. Los resultados de este notebook sustentan las respuestas; los textos son propios.

### Fase 1. Exploración y calidad de los datos

Distribuciones, nulos, duplicados y valores extremos con la decisión tomada sobre cada uno; correlación con el objetivo y matriz de correlación entre variables explicativas, con una lectura sobre redundancia.

### Fase 2. Preparación de variables y control de fuga

Tabla de control de variables completa y lista `fuera_del_modelo` justificada. Tabla que compara el mismo modelo con todas las columnas y sin las excluidas, con MAE, RMSE y R². En 100-150 palabras: por qué cada columna excluida no es válida en el momento de la decisión, en qué se diferencian las formas de fuga encontradas y por qué una métrica muy alta, por sí sola, no valida el análisis.

### Fase 3. Modelado y registro comparativo

`reportes/registro_semana9.csv` con todas las corridas: líneas base, los tres algoritmos, al menos una idea propia de *feature engineering* y la comparación entre partición temporal y aleatoria. Justifique la partición elegida.

### Fase 4. Evaluación e interpretación de métricas

Qué modelo selecciona y cuánto mejora frente a las líneas base, en puntos de popularidad; en qué género es menos confiable y hacia dónde se equivoca. 100-150 palabras que relacionen las métricas con la decisión de la disquera.

### Fase 5. Importancia de características y conclusión de negocio

Importancia por permutación del modelo seleccionado y, si es lineal, sus coeficientes. Conclusión de negocio en 100-150 palabras: qué atributos hacen popular a una canción según el modelo, con sus límites (asociación, no causa).

### Auditoría de un resultado generado por IA

Pida a un asistente de IA que resuelva esta misma tarea con un encargo escrito por usted. Entregue el encargo, la respuesta y el código tal como los recibió. Aplique las cinco preguntas y entregue la ficha de auditoría con al menos cuatro afirmaciones del asistente, cada una con su evidencia ejecutable y su veredicto. Cierre con el memo de cinco líneas para la disquera: decisión, estimación, validez, incertidumbre y siguiente paso.

### Rúbrica

| Criterio | Qué se revisa | Peso |
|---|---|---|
| Exploración y calidad de los datos | Nulos, duplicados, valores extremos con decisión explícita, correlaciones y redundancia | 15 % |
| Preparación de variables y control de fuga | Tabla de control, exclusiones justificadas con evidencia, preprocesamiento dentro del `Pipeline`, partición temporal | 20 % |
| Modelado y registro comparativo | Líneas base, tres algoritmos, idea propia de *feature engineering*, registro completo con su validación | 20 % |
| Interpretación de métricas | Lectura crítica de MAE, RMSE, R² y sesgo, por segmento y frente a la línea base; no solo R² | 15 % |
| Importancia de características y conclusión de negocio | Lectura correcta de la importancia y los coeficientes; conclusión útil, con límites | 10 % |
| Auditoría de la salida de IA | Rigor de la validación: cada veredicto con evidencia ejecutable; memo de cinco líneas | 20 % |

Afirmar que un modelo es mejor porque tiene mayor R² no constituye una justificación suficiente. La comparación debe relacionar las métricas con la forma en que se validó, con los tipos de error y con la decisión que se quiere tomar.